# Phase 2 - Cleaning (raw -> silver), column by column
Raw files are never edited. Every column that needs a fix has its own cell with a before/after check.
Run all cells top to bottom with `HCL_Hackathon` as the working directory.

Outputs: `silver/*.csv`, `silver/silver.db`, `silver/rejects.csv`, `before_after_counts.csv`, `02_cleaning_report.md`.

## 0. Setup and helper functions

In [ ]:
import pandas as pd, numpy as np, re, json, sqlite3
from pathlib import Path
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

BASE = Path.cwd()
SILVER = BASE / 'silver'; SILVER.mkdir(exist_ok=True)
NULL_TOKENS = {'', 'na', 'n/a', 'none', 'null', 'nan', 'unknown', '-'}
rejects, fixes, raw_stats, dstats, before_distinct = [], [], {}, {}, {}

def rd(name):
    """read csv with everything as text, trimmed"""
    df = pd.read_csv(BASE / f'{name}.csv', dtype=str, keep_default_na=False, encoding='utf-8')
    return df.apply(lambda s: s.str.strip())

def nullify(s):
    return s.mask(s.str.lower().isin(NULL_TOKENS))

def nulls_raw(df):
    return int(df.apply(lambda c: c.astype(str).str.strip().str.lower().isin(NULL_TOKENS)).sum().sum())

def fix(table, what, n):
    fixes.append((table, what, int(n)))
    print(f'  -> {n} rows: {what}')

def compare(name, before, after, n=15):
    print(f'{name}: distinct {before.nunique(dropna=False)} -> {after.nunique(dropna=False)}')
    print(after.value_counts(dropna=False).head(n).to_string())

def log(table, df, key, reason, action):
    for _, r in df.iterrows():
        rec = {k: (None if pd.isna(v) else str(v)) for k, v in r.items() if not str(k).startswith('_')}
        rejects.append(dict(table=table, record_key=rec.get(key), reason=reason, action=action,
                            record_json=json.dumps(rec, ensure_ascii=False)))

def _pd(v):
    """one date string -> (Timestamp or NaT, is_ambiguous)"""
    if not isinstance(v, str): return (pd.NaT, False)
    s = v.strip()
    try:
        if re.fullmatch(r'\d{4}-\d{2}-\d{2}', s): return (pd.Timestamp(s), False)
        if re.fullmatch(r'\d{2}-[A-Za-z]{3}-\d{4}', s): return (pd.to_datetime(s, format='%d-%b-%Y'), False)
        m = re.fullmatch(r'(\d{2})/(\d{2})/(\d{4})', s)
        if m:
            a, b, y = map(int, m.groups())
            if a > 12: return (pd.Timestamp(y, b, a), False)   # DD/MM/YYYY
            if b > 12: return (pd.Timestamp(y, a, b), False)   # MM/DD/YYYY
            return (pd.Timestamp(y, a, b), a != b)             # ambiguous -> MM/DD default
    except Exception:
        pass
    return (pd.NaT, False)

def parse_dates(s):
    t = s.map(_pd)
    return pd.to_datetime(t.map(lambda x: x[0])), t.map(lambda x: x[1]).astype(bool)

def parse_amount(s):
    def f(v):
        if not isinstance(v, str): return np.nan
        t = re.sub(r'[^\d.\-]', '', v)
        return float(t) if re.fullmatch(r'-?\d+(\.\d+)?', t) else np.nan
    return s.map(f).astype(float)

BOOL = {'y': True, 'yes': True, 'true': True, '1': True, 'n': False, 'no': False, 'false': False, '0': False}
def to_bool(s):
    return s.astype(str).str.strip().str.lower().map(BOOL).astype(object)

CITY = {'mumbai': 'Mumbai', 'bangalore': 'Bangalore', 'bengaluru': 'Bangalore', 'delhi': 'Delhi', 'new delhi': 'Delhi'}
def city(s):
    return s.str.lower().map(CITY).fillna(s.str.title())

def dedupe(df, key, table, prefer=None):
    """exact duplicates -> dropped; same key but different values -> winner = prefer cols, then most complete row"""
    df = df.copy()
    cols = [c for c in df.columns if not c.startswith('_')]
    df['_score'] = df[cols].notna().sum(axis=1)
    exact = df.duplicated(subset=cols, keep='first')
    log(table, df[exact], key, 'EXACT_DUPLICATE', 'dropped')
    df = df[~exact]
    df = df.sort_values((prefer or []) + ['_score'], ascending=False, kind='stable')
    conf = df.duplicated(subset=[key], keep='first')
    log(table, df[conf], key, 'CONFLICTING_DUPLICATE', 'dropped')
    df = df[~conf].sort_index().drop(columns='_score')
    dstats[table] = dict(exact=int(exact.sum()), conf=int(conf.sum()))
    print(f'  -> {int(exact.sum())} exact duplicates dropped, {int(conf.sum())} conflicting duplicates dropped')
    return df

def drop_cols(df):
    return df[[c for c in df.columns if not c.startswith('_')]].reset_index(drop=True)

def reject_where(df, mask, key, table, reason):
    print(f'  -> {int(mask.sum())} rows quarantined: {reason}')
    log(table, df[mask], key, reason, 'quarantined')
    return df[~mask]

## 1. suppliers (JSON)

In [ ]:
sup = pd.DataFrame(json.load(open(BASE / 'suppliers_json.txt', encoding='utf-8'))).astype(object)
raw_stats['suppliers'] = dict(raw=len(sup), nulls=int(sup.isna().sum().sum()) + int((sup.country == 'Unknown').sum()))
print(len(sup), 'rows'); sup.head(20)

**country** - None / "Unknown" are missing values, make them real NULL.

In [ ]:
before = sup.country.copy()
sup['country'] = sup.country.map(lambda v: np.nan if v is None or str(v).strip().lower() in NULL_TOKENS else str(v).strip())
compare('country', before, sup.country)

**rating** - values above 5 are on a 0-10 scale, halve them so everything is 0-5. Raw value and original scale are kept.

In [ ]:
sup['rating_raw'] = pd.to_numeric(sup.rating, errors='coerce')
sup['rating_scale_raw'] = np.where(sup.rating_raw > 5, '0-10', '0-5')
sup['rating'] = np.where(sup.rating_raw > 5, sup.rating_raw / 2, sup.rating_raw).round(2)
fix('suppliers', 'rating >5 treated as 0-10 scale and halved', (sup.rating_raw > 5).sum())
print('max rating before', sup.rating_raw.max(), '-> after', sup.rating.max())

**is_preferred** - "N", "Y", "true", False, True ... all become one boolean.

In [ ]:
before = sup.is_preferred.astype(str)
sup['is_preferred'] = to_bool(sup.is_preferred.map(lambda v: str(v)))
compare('is_preferred', before, sup.is_preferred)

**contract_start_date** - mix of `YYYY-MM-DD` and `DD-Mon-YYYY`, both unambiguous.

In [ ]:
sup['contract_start_date'], sup['contract_date_ambiguous'] = parse_dates(sup.contract_start_date.astype(str))
print('unparseable:', sup.contract_start_date.isna().sum(), '| ambiguous:', sup.contract_date_ambiguous.sum())

**duplicate supplier_id S004** - two different records. Winner rule: most complete record (a real country beats "Unknown").

In [ ]:
print(sup[sup.supplier_id == 'S004'][['supplier_id', 'supplier_name', 'country', 'rating']])
sup = dedupe(sup, 'supplier_id', 'suppliers')
print(sup[sup.supplier_id == 'S004'][['supplier_id', 'supplier_name', 'country', 'rating']])
suppliers = drop_cols(sup); print(len(suppliers), 'suppliers in silver')

## 2. customers

In [ ]:
cu_raw = rd('customers')
raw_stats['customers'] = dict(raw=len(cu_raw), nulls=nulls_raw(cu_raw))
before_distinct['customers'] = {k: cu_raw[k].nunique() for k in ['customer_segment', 'city', 'is_active']}
cu = cu_raw.copy()
for col in cu.columns: cu[col] = nullify(cu[col])      # NA / N/A / unknown / blank -> real NULL
print(len(cu), 'rows | null counts:'); print(cu.isna().sum()[lambda s: s > 0])

**email** - same email written in upper and lower case (this is what makes the duplicate customer rows differ).

In [ ]:
before = cu.email.copy()
cu['email'] = cu.email.str.lower()
print((before != cu.email).sum(), 'emails lower-cased')

**customer_segment** - 15 spellings (Glod, Broze, Silverr, GOLD...) -> Bronze / Silver / Gold / Platinum.

In [ ]:
SEG = {'glod': 'Gold', 'gold': 'Gold', 'broze': 'Bronze', 'bronze': 'Bronze', 'silver': 'Silver', 'silverr': 'Silver', 'platinum': 'Platinum'}
before = cu.customer_segment.copy()
cu['customer_segment'] = cu.customer_segment.str.lower().map(SEG)
compare('customer_segment', before, cu.customer_segment)

**city** - Bengaluru / BANGALORE / bangalore -> Bangalore, New Delhi / DELHI -> Delhi, etc. (Bangalore matches the spelling in orders.shipping_city).

In [ ]:
before = cu.city.copy()
cu['city'] = city(cu.city)
compare('city', before, cu.city)

**state** - already consistent, only title-case for safety.

In [ ]:
before = cu.state.copy()
cu['state'] = cu.state.str.title()
compare('state', before, cu.state)

**phone** - 5 formats. Keep digits only, drop +91 / leading 0, result must be 10 digits else NULL.

In [ ]:
def clean_phone(v):
    if pd.isna(v): return np.nan
    d = re.sub(r'\D', '', v)
    if len(d) == 12 and d.startswith('91'): d = d[2:]
    elif len(d) == 11 and d.startswith('0'): d = d[1:]
    return d if len(d) == 10 else np.nan
before = cu.phone.copy()
cu['phone'] = cu.phone.map(clean_phone)
fix('customers', 'phone not a valid 10-digit number -> NULL', (cu.phone.isna() & before.notna()).sum())
print(cu.phone.str.len().value_counts(dropna=False).to_string())

**zip_code** - must be exactly 6 digits, otherwise NULL (values like "N/A").

In [ ]:
before = cu.zip_code.copy()
cu['zip_code'] = cu.zip_code.where(cu.zip_code.str.fullmatch(r'\d{6}', na=False))
fix('customers', 'zip not 6 digits -> NULL', (cu.zip_code.isna() & before.notna()).sum())

**is_active** - Y / yes / true / 1 / True -> True, N / no / False / 0 -> False.

In [ ]:
before = cu.is_active.copy()
cu['is_active'] = to_bool(cu.is_active)
compare('is_active', before, cu.is_active)

**signup_date** - 3 formats. Rule: ISO and DD-Mon-YYYY are exact; for dd/mm/yyyy a part above 12 shows the order; if both parts are 12 or below it is ambiguous -> MM/DD, and a flag column is kept.

In [ ]:
cu['signup_date'], cu['signup_date_ambiguous'] = parse_dates(cu.signup_date)
fix('customers', 'ambiguous dd/mm vs mm/dd dates parsed as MM/DD', cu.signup_date_ambiguous.sum())
print('unparseable / missing:', cu.signup_date.isna().sum())

**duplicate customer_id** - after lower-casing email the 5 duplicate pairs are identical, so they are exact duplicates.

In [ ]:
cu = dedupe(cu, 'customer_id', 'customers')
customers = drop_cols(cu); print(len(customers), 'customers in silver, unique ids:', customers.customer_id.is_unique)

## 3. products

In [ ]:
pr = rd('products'); oi_raw = rd('order_items')
raw_stats['products'] = dict(raw=len(pr), nulls=nulls_raw(pr))
before_distinct['products'] = {k: pr[k].nunique() for k in ['category', 'is_active']}
oi_price = pd.to_numeric(oi_raw.unit_price, errors='coerce')
price_seen = set(zip(oi_raw.product_id, oi_price.round(2)))     # prices really used on order lines
for col in pr.columns: pr[col] = nullify(pr[col])
print(len(pr), 'rows | null counts:'); print(pr.isna().sum()[lambda s: s > 0])

**unit_price** - strip the rupee symbol / commas, convert to number. Prices <= 0 are invalid -> NULL (fixed later).

In [ ]:
before = pr.unit_price.copy()
pr['unit_price'] = parse_amount(pr.unit_price)
pr['_price_valid'] = pr.unit_price > 0
print('invalid (<=0 or unreadable):', (~pr._price_valid).sum())
pr['unit_price'] = pr.unit_price.where(pr._price_valid)
pr['_in_orders'] = [(a, round(b, 2)) in price_seen if pd.notna(b) else False for a, b in zip(pr.product_id, pr.unit_price)]
print('price also seen on an order line:', pr._in_orders.sum())

**cost_price** - already numeric, convert from text.

In [ ]:
pr['cost_price'] = pd.to_numeric(pr.cost_price, errors='coerce')
print(pr.cost_price.describe())

**product_name** - some rows end with "(Revised)". Remember which are revised (used to pick a winner), then remove the suffix.

In [ ]:
pr['_revised'] = pr.product_name.str.contains(r'\(Revised\)\s*$', na=False)
pr['product_name'] = pr.product_name.str.replace(r'\s*\(Revised\)\s*$', '', regex=True)
print(pr._revised.sum(), 'revised rows; names missing:', pr.product_name.isna().sum())

**category** - Home and Kitchen / home & kitchen -> Home & Kitchen, Electornics -> Electronics, case fixed.

In [ ]:
CAT = {'home and kitchen': 'Home & Kitchen', 'home & kitchen': 'Home & Kitchen', 'electornics': 'Electronics'}
before = pr.category.copy()
pr['category'] = pr.category.str.lower().map(CAT).fillna(pr.category.str.title())
compare('category', before, pr.category)

**is_active**

In [ ]:
before = pr.is_active.copy()
pr['is_active'] = to_bool(pr.is_active)
compare('is_active', before, pr.is_active)

**launch_date**

In [ ]:
pr['launch_date'], pr['launch_date_ambiguous'] = parse_dates(pr.launch_date)
fix('products', 'ambiguous dd/mm vs mm/dd dates parsed as MM/DD', pr.launch_date_ambiguous.sum())

**duplicate product_id (5 ids, conflicting price)** - winner rule in order: (1) valid price, (2) price that appears on real order lines, (3) the "(Revised)" row as the latest, (4) most complete row.

In [ ]:
dups = pr[pr.product_id.duplicated(keep=False)].sort_values('product_id')
print(dups[['product_id', 'product_name', 'unit_price', '_price_valid', '_in_orders', '_revised']])
pr = dedupe(pr, 'product_id', 'products', prefer=['_price_valid', '_in_orders', '_revised'])
print(pr[pr.product_id.isin(dups.product_id)][['product_id', 'product_name', 'unit_price']])

**missing category** - fill from the sub_category (each sub_category belongs to one category).

In [ ]:
submap = pr.dropna(subset=['category']).groupby('sub_category').category.agg(lambda s: s.mode().iat[0])
miss = pr.category.isna()
pr.loc[miss, 'category'] = pr.loc[miss, 'sub_category'].map(submap)
pr['category_inferred'] = miss & pr.category.notna()
fix('products', 'blank category inferred from sub_category', pr.category_inferred.sum())
print('still missing:', pr.category.isna().sum())

**invalid unit_price left after de-duplication** - use the most common price seen on that product's order lines.

In [ ]:
modal = oi_raw.assign(pr=oi_price)[lambda d: d.pr > 0].groupby('product_id').pr.agg(lambda s: s.mode().iat[0])
mp = pr.unit_price.isna()
pr.loc[mp, 'unit_price'] = pr.loc[mp, 'product_id'].map(modal)
pr['price_imputed'] = mp & pr.unit_price.notna()
fix('products', 'invalid price replaced by modal order-line price', pr.price_imputed.sum())
print('still missing price:', pr.unit_price.isna().sum())

**negative margin** - cost above price is a real business signal, so flag it, do not change it.

In [ ]:
pr['negative_margin'] = pr.cost_price > pr.unit_price
print(pr.negative_margin.sum(), 'products sell below cost')

**supplier_id** - ids not present in suppliers (S92, S93, S96, S98) are orphans. Keep the product, set supplier_id NULL, keep the raw value, log it in rejects.

In [ ]:
orph = ~pr.supplier_id.isin(set(suppliers.supplier_id))
log('products', pr[orph], 'product_id', 'ORPHAN_SUPPLIER', 'flagged_kept')
pr['supplier_id_raw'] = pr.supplier_id
pr.loc[orph, 'supplier_id'] = np.nan
print(orph.sum(), 'products with unknown supplier')
products = drop_cols(pr); print(len(products), 'products in silver')

## 4. orders

In [ ]:
od = rd('orders')
raw_stats['orders'] = dict(raw=len(od), nulls=nulls_raw(od))
before_distinct['orders'] = {k: od[k].nunique() for k in ['order_status', 'channel', 'promo_code']}
for col in od.columns: od[col] = nullify(od[col])
print(len(od), 'rows')

**order_status** - Complted / COMPLETE / completed ..., Canceled / Cancled / cancelled ... -> Completed, Returned, Pending, Cancelled.

In [ ]:
def status(v):
    if pd.isna(v): return np.nan
    v = v.lower()
    for k, n in [('comp', 'Completed'), ('ret', 'Returned'), ('pend', 'Pending'), ('canc', 'Cancelled')]:
        if v.startswith(k): return n
    return np.nan
before = od.order_status.copy()
od['order_status'] = od.order_status.map(status)
compare('order_status', before, od.order_status)

**channel** - app / App / APP -> App, web -> Web, store / In-store / Instore -> Store.

In [ ]:
before = od.channel.copy()
od['channel'] = od.channel.str.lower().map(lambda v: np.nan if pd.isna(v) else ('App' if v == 'app' else 'Web' if v == 'web' else 'Store'))
compare('channel', before, od.channel)

**promo_code** - NA / N/A / None / null / blank all mean "no promo" -> NULL; real codes upper-cased.

In [ ]:
before = od.promo_code.copy()
od['promo_code'] = od.promo_code.str.upper()
print('orders without promo:', od.promo_code.isna().sum(), '| distinct codes:', before.nunique(), '->', od.promo_code.nunique())

**payment_method** - already 5 clean values, no change needed.

In [ ]:
print(od.payment_method.value_counts(dropna=False).to_string())

**shipping_city** - same city mapping as customers.

In [ ]:
before = od.shipping_city.copy()
od['shipping_city'] = city(od.shipping_city)
compare('shipping_city', before, od.shipping_city)

**order_date**

In [ ]:
od['order_date'], od['order_date_ambiguous'] = parse_dates(od.order_date)
fix('orders', 'ambiguous dd/mm vs mm/dd dates parsed as MM/DD', od.order_date_ambiguous.sum())
print('range:', od.order_date.min().date(), 'to', od.order_date.max().date(), '| unparseable:', od.order_date.isna().sum())

**duplicate order_id** - 3 exact duplicates.

In [ ]:
od = dedupe(od, 'order_id', 'orders')

**orphan customer_id** - the order exists but the customer is not in the customers table. Keep (revenue is real), flag `customer_orphan`, log in rejects. KPI 2 (customer level) excludes these.

In [ ]:
od['customer_orphan'] = ~od.customer_id.isin(customers.customer_id)
log('orders', od[od.customer_orphan], 'order_id', 'ORPHAN_CUSTOMER', 'flagged_kept')
print(od.customer_orphan.sum(), 'orders flagged')
orders = drop_cols(od); print(len(orders), 'orders in silver')

## 5. order_items

In [ ]:
oi = rd('order_items')
raw_stats['order_items'] = dict(raw=len(oi), nulls=nulls_raw(oi))
for col in oi.columns: oi[col] = nullify(oi[col])
for col in ['quantity', 'unit_price', 'discount_pct', 'line_total']: oi[col] = pd.to_numeric(oi[col], errors='coerce')
oi = oi.rename(columns={'line_total': 'line_total_raw'})      # keep stored value, never overwrite it
print(len(oi), 'rows'); oi.describe().T

**duplicate order_item_id** - 5 exact duplicates.

In [ ]:
oi = dedupe(oi, 'order_item_id', 'order_items')

**order_id** - item whose order does not exist cannot be reported -> quarantine.

In [ ]:
oi = reject_where(oi, ~oi.order_id.isin(orders.order_id), 'order_item_id', 'order_items', 'ORPHAN_ORDER')

**product_id** - item whose product does not exist -> quarantine.

In [ ]:
oi = reject_where(oi, ~oi.product_id.isin(products.product_id), 'order_item_id', 'order_items', 'ORPHAN_PRODUCT')

**quantity** - zero or negative quantity is invalid -> quarantine.

In [ ]:
print(oi.quantity.value_counts().sort_index().to_string())
oi = reject_where(oi, ~(oi.quantity > 0), 'order_item_id', 'order_items', 'INVALID_QUANTITY')

**unit_price** - missing or negative -> take the product list price from silver products.

In [ ]:
bad = oi.unit_price.isna() | (oi.unit_price <= 0)
oi['price_imputed'] = bad
oi.loc[bad, 'unit_price'] = oi.loc[bad, 'product_id'].map(products.set_index('product_id').unit_price)
fix('order_items', 'missing/negative unit_price replaced by product list price', bad.sum())
oi = reject_where(oi, oi.unit_price.isna() | (oi.unit_price <= 0), 'order_item_id', 'order_items', 'UNPRICEABLE')

**discount_pct** - valid range is 0-100; negative or missing -> 0 and flagged.

In [ ]:
print('before:', sorted(oi.discount_pct.dropna().unique()))
badd = oi.discount_pct.isna() | (oi.discount_pct < 0) | (oi.discount_pct > 100)
oi['discount_invalid'] = badd
oi.loc[badd, 'discount_pct'] = 0
fix('order_items', 'discount outside 0-100 -> 0', badd.sum())
print('after:', sorted(oi.discount_pct.unique()))

**line_total** - stored value does not always equal quantity x price x (1 - discount). Recompute from inputs and **trust the recomputed value**; keep the stored one as `line_total_raw` with a mismatch flag.

In [ ]:
oi['line_total'] = (oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100)).round(2)
oi['line_total_mismatch'] = oi.line_total_raw.isna() | ((oi.line_total - oi.line_total_raw).abs() > 0.01)
fix('order_items', 'stored line_total did not reconcile; recomputed value trusted', oi.line_total_mismatch.sum())
print('stored total:', round(oi.line_total_raw.sum(), 2), '| recomputed total:', round(oi.line_total.sum(), 2))
order_items = drop_cols(oi); print(len(order_items), 'order_items in silver')
oi[oi.line_total_mismatch][['order_item_id', 'quantity', 'unit_price', 'discount_pct', 'line_total_raw', 'line_total']].head(10)

## 6. payments

In [ ]:
pm = rd('payments')
raw_stats['payments'] = dict(raw=len(pm), nulls=nulls_raw(pm))
before_distinct['payments'] = {'payment_status': pm.payment_status.nunique()}
for col in pm.columns: pm[col] = nullify(pm[col])
print(len(pm), 'rows')

**amount** - formats: `67330.43`, `₹210,054.96`, `$181,349.01`, `73078.38 INR`. Remember which symbol was used, then strip everything but the number. **No FX conversion for $** (checked at the end of this section).

In [ ]:
pm['amount_currency_raw'] = np.where(pm.amount.str.contains('$', regex=False, na=False), 'USD',
                             np.where(pm.amount.str.contains('\u20b9', regex=False, na=False), 'INR', 'none/INR'))
pm['amount'] = parse_amount(pm.amount)
fix('payments', 'amounts with $ symbol (no FX conversion applied)', (pm.amount_currency_raw == 'USD').sum())
print(pm.amount_currency_raw.value_counts().to_string()); print('unreadable amounts:', pm.amount.isna().sum())

**payment_status** - Succeded / success / SUCCESS ... -> Success, Pending, Failed, Refunded.

In [ ]:
def pstat(v):
    if pd.isna(v): return np.nan
    v = v.lower()
    for k, n in [('succ', 'Success'), ('pend', 'Pending'), ('fail', 'Failed'), ('refund', 'Refunded')]:
        if v.startswith(k): return n
    return np.nan
before = pm.payment_status.copy()
pm['payment_status'] = pm.payment_status.map(pstat)
compare('payment_status', before, pm.payment_status)

**payment_method** - already clean (5 values).

In [ ]:
print(pm.payment_method.value_counts(dropna=False).to_string())

**payment_date** - 10 blanks stay NULL.

In [ ]:
pm['payment_date'], pm['payment_date_ambiguous'] = parse_dates(pm.payment_date)
fix('payments', 'ambiguous dd/mm vs mm/dd dates parsed as MM/DD', pm.payment_date_ambiguous.sum())
print('missing dates:', pm.payment_date.isna().sum())

**duplicate payment_id** - 4 exact duplicates.

In [ ]:
pm = dedupe(pm, 'payment_id', 'payments')

**order_id** - payment for an order that does not exist -> quarantine.

In [ ]:
pm = reject_where(pm, ~pm.order_id.isin(orders.order_id), 'payment_id', 'payments', 'ORPHAN_ORDER')
payments = drop_cols(pm); print(len(payments), 'payments in silver')

**check: are $ amounts really dollars?** Compare Success payments with the order total (recomputed). If $ rows match as often as the others, they are INR with a wrong symbol.

In [ ]:
tot = order_items.groupby('order_id').line_total.sum()
d = payments[payments.payment_status == 'Success'].copy()
d['order_total'] = d.order_id.map(tot)
d['match'] = ((d.amount - d.order_total).abs() <= 0.01 * d.order_total)
fx_diag = d.groupby('amount_currency_raw').match.agg(['mean', 'count']).round(3)
fx_diag

## 7. returns

In [ ]:
rt = rd('returns')
raw_stats['returns'] = dict(raw=len(rt), nulls=nulls_raw(rt))
before_distinct['returns'] = {k: rt[k].nunique() for k in ['return_reason', 'return_status']}
for col in rt.columns: rt[col] = nullify(rt[col])
print(len(rt), 'rows')

**return_reason** - case variants and the typo "Damged".

In [ ]:
before = rt.return_reason.copy()
rt['return_reason'] = rt.return_reason.str.lower().map({'damged': 'Damaged', 'damaged': 'Damaged'}).fillna(rt.return_reason.str.title())
compare('return_reason', before, rt.return_reason)

**return_status** - Approved / Rejected / Pending.

In [ ]:
before = rt.return_status.copy()
rt['return_status'] = rt.return_status.str.title()
compare('return_status', before, rt.return_status)

**refund_amount** - to number; 4 blanks stay NULL (not guessed).

In [ ]:
rt['refund_amount'] = parse_amount(rt.refund_amount)
print('missing refunds:', rt.refund_amount.isna().sum())

**return_date**

In [ ]:
rt['return_date'], rt['return_date_ambiguous'] = parse_dates(rt.return_date)
fix('returns', 'ambiguous dd/mm vs mm/dd dates parsed as MM/DD', rt.return_date_ambiguous.sum())

**duplicate return_id** - 3 exact duplicates.

In [ ]:
rt = dedupe(rt, 'return_id', 'returns')

**order_item_id** - return of an item that is not in silver order_items -> quarantine.

In [ ]:
rt = reject_where(rt, ~rt.order_item_id.isin(order_items.order_item_id), 'return_id', 'returns', 'ORPHAN_ORDER_ITEM')

**sanity flags** (flag only, nothing is deleted): refund missing, refund larger than the item line total, return dated before the order.

In [ ]:
it = order_items.set_index('order_item_id')
rt['refund_missing'] = rt.refund_amount.isna()
rt['refund_exceeds_line_total'] = rt.refund_amount > rt.order_item_id.map(it.line_total) + 0.01
odate = rt.order_item_id.map(it.order_id).map(orders.set_index('order_id').order_date)
rt['return_before_order'] = rt.return_date < odate
fix('returns', 'refund_amount blank (kept NULL, flagged)', rt.refund_missing.sum())
fix('returns', 'refund greater than recomputed line_total (flagged)', rt.refund_exceeds_line_total.sum())
fix('returns', 'return_date earlier than order_date (flagged)', rt.return_before_order.sum())
returns = drop_cols(rt); print(len(returns), 'returns in silver')

## 8. Integrity checks

In [ ]:
assert order_items.order_id.isin(orders.order_id).all()
assert order_items.product_id.isin(products.product_id).all()
assert payments.order_id.isin(orders.order_id).all()
assert returns.order_item_id.isin(order_items.order_item_id).all()
for t, k in [(customers, 'customer_id'), (products, 'product_id'), (suppliers, 'supplier_id'), (orders, 'order_id'),
             (order_items, 'order_item_id'), (payments, 'payment_id'), (returns, 'return_id')]:
    assert t[k].is_unique, k
print('all foreign keys resolve and all primary keys are unique')

## 9. Write silver tables and rejects

In [ ]:
tables = dict(customers=customers, products=products, suppliers=suppliers, orders=orders,
              order_items=order_items, payments=payments, returns=returns)
def prep(df):
    df = df.copy()
    for col in df.columns:
        if str(df[col].dtype).startswith('datetime'): df[col] = df[col].dt.strftime('%Y-%m-%d')
    return df
con = sqlite3.connect(SILVER / 'silver.db')
for n, df in tables.items():
    q = prep(df); q.to_csv(SILVER / f'{n}.csv', index=False, encoding='utf-8')
    s = q.copy()
    for col in s.columns:
        nn = s[col].dropna()
        if s[col].dtype == object and len(nn) and all(isinstance(x, (bool, np.bool_)) for x in nn):
            s[col] = s[col].map({True: 1, False: 0}).astype('Int64')
        elif s[col].dtype == bool: s[col] = s[col].astype(int)
    s.to_sql(n, con, if_exists='replace', index=False)
rej = pd.DataFrame(rejects, columns=['table', 'record_key', 'reason', 'action', 'record_json'])
rej.to_csv(SILVER / 'rejects.csv', index=False, encoding='utf-8')
rej.to_sql('rejects', con, if_exists='replace', index=False); con.close()
print('written to', SILVER)

## 10. Before / after counts and report

In [ ]:
rows = []
for n, df in tables.items():
    r = rej[rej.table == n]
    rows.append(dict(table=n, raw_rows=raw_stats[n]['raw'], exact_dups_dropped=dstats[n]['exact'],
                     conflicting_dups_dropped=dstats[n]['conf'],
                     quarantined=int((r.action == 'quarantined').sum()), flagged_kept=int((r.action == 'flagged_kept').sum()),
                     silver_rows=len(df), raw_null_cells=raw_stats[n]['nulls'], silver_null_cells=int(df.isna().sum().sum())))
ba = pd.DataFrame(rows); ba.to_csv(BASE / 'before_after_counts.csv', index=False)
ba

In [ ]:
drows = []
for t, cols in before_distinct.items():
    for col, nb in cols.items():
        drows.append(dict(table=t, column=col, distinct_before=nb, distinct_after=tables[t][col].nunique(),
                          values_after=', '.join(sorted(map(str, tables[t][col].dropna().unique())))[:80]))
dd = pd.DataFrame(drows)
fx = pd.DataFrame(fixes, columns=['table', 'fix', 'rows_affected'])
rs = rej.groupby(['table', 'reason', 'action']).size().reset_index(name='rows')
display(dd); display(fx); display(rs)

In [ ]:
def md_table(df): return '| ' + ' | '.join(map(str, df.columns)) + ' |\n|' + '---|' * len(df.columns) + '\n' + ''.join('| ' + ' | '.join(map(str, r)) + ' |\n' for r in df.itertuples(index=False))
RULES = """
## Cleaning rules (decided before coding)
**Text / labels** - trim, lower-case compare, map to one canonical label; typos fixed with explicit maps (Glod->Gold, Broze->Bronze, Silverr->Silver, Complted->Completed, Canceled/Cancled->Cancelled, Succeded->Success, Electornics->Electronics, Damged->Damaged). Channels: app->App, web->Web, store/In-store/Instore->Store. Cities: Bengaluru/BANGALORE->Bangalore, New Delhi->Delhi. Null tokens (blank, NA, N/A, None, null, unknown, '-') become real NULL; promo NULL = no promo.
**Booleans** - Y/yes/true/1 -> TRUE, N/no/false/0 -> FALSE.
**Dates** - ISO and DD-Mon-YYYY are exact. dd/mm/yyyy: first part >12 -> DD/MM, second part >12 -> MM/DD, both <=12 -> ambiguous, default MM/DD with a `*_date_ambiguous` flag kept.
**Amounts** - strip symbols, commas and ' INR'. $ amounts: no FX conversion (magnitudes match INR rows, see check); marker kept in `amount_currency_raw`.
**Phone / zip** - phone to 10 digits else NULL; zip must be 6 digits else NULL.
**Exact duplicates** - dropped and logged.
**Conflicting duplicates** - customers: most complete (after email lower-casing they are exact dups); suppliers S004: most complete (real country beats Unknown); products: valid price, then price seen on order lines, then '(Revised)' row, then most complete.
**Orphans** - child rows without a parent are quarantined in rejects (items->orders/products, payments->orders, returns->items, cascading in that order). Orders with unknown customer are kept and flagged (customer_orphan); revenue KPIs include them, KPI 2 excludes them. Products with unknown supplier keep the row with supplier_id NULL.
**line_total** - recomputed = qty x unit_price x (1 - discount/100); recomputed value trusted, stored value kept as line_total_raw with mismatch flag.
**Suppliers rating** - values >5 are 0-10 scale, halved to 0-5 (a 10-scale value <=5 cannot be detected).
**Returns** - blank refund stays NULL; refund > line_total and return before order are flagged, not deleted.
"""
rep = f"# Phase 2 - Cleaning report\n{RULES}\n## Before / after row counts\n{md_table(ba)}\n## Distinct values before / after\n{md_table(dd)}\n## Fixes applied\n{md_table(fx)}\n## Rejects summary\n{md_table(rs)}\n## $ amount check (Success payments: share where amount == order total within 1%, no conversion)\n{md_table(fx_diag.reset_index())}\nFiles: silver/*.csv, silver/silver.db, silver/rejects.csv\n"
(BASE / '02_cleaning_report.md').write_text(rep, encoding='utf-8')
print('02_cleaning_report.md written')